In [2]:
# Cell 1: Backtest notebook setup

import sys
from pathlib import Path

import ee
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

EE_PROJECT = "august-analyze"

try:
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine initialized with project: {EE_PROJECT}")
except Exception:
    print("Earth Engine not initialized. Running authentication...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine authenticated and initialized with project: {EE_PROJECT}")

print("Project root:", PROJECT_ROOT)
print("Notebook purpose: clean ADM2 drought backtesting")

Earth Engine initialized with project: august-analyze
Project root: C:\Projects\Infer RozviDrought\RozviDrought
Notebook purpose: clean ADM2 drought backtesting


In [3]:
# Cell 2: Load Zimbabwe ADM2 administrative polygons from Earth Engine

ADMIN_LEVEL = 2
GAUL_ASSET = f"FAO/GAUL/2015/level{ADMIN_LEVEL}"

admin_fc = (
    ee.FeatureCollection(GAUL_ASSET)
    .filter(ee.Filter.eq("ADM0_NAME", "Zimbabwe"))
)

admin_count = admin_fc.size().getInfo()
first_admin = admin_fc.first().toDictionary().getInfo()

print("Loaded admin layer:", GAUL_ASSET)
print("Country: Zimbabwe")
print("Admin level:", f"ADM{ADMIN_LEVEL}")
print("Polygon count:", admin_count)
print("Example properties:")
print(first_admin)

Loaded admin layer: FAO/GAUL/2015/level2
Country: Zimbabwe
Admin level: ADM2
Polygon count: 62
Example properties:
{'ADM0_CODE': 271, 'ADM0_NAME': 'Zimbabwe', 'ADM1_CODE': 3436, 'ADM1_NAME': 'Harare', 'ADM2_CODE': 68807, 'ADM2_NAME': 'Chitungwiza', 'DISP_AREA': 'NO', 'EXP2_YEAR': 3000, 'STATUS': 'Member State', 'STR2_YEAR': 2006, 'Shape_Area': 0.00407260399874, 'Shape_Leng': 0.34197890796}


In [4]:
# Cell 3: Convert Earth Engine ADM2 polygons to local Shapely geometries

from shapely.geometry import shape

print("Downloading ADM2 features from Earth Engine...")
admin_features = admin_fc.getInfo()["features"]

admin_records = []

for feature in admin_features:
    props = feature["properties"]
    geom = feature["geometry"]

    admin_records.append({
        "adm0_name": props.get("ADM0_NAME"),
        "adm1_name": props.get("ADM1_NAME"),
        "adm2_name": props.get("ADM2_NAME"),
        "adm2_code": props.get("ADM2_CODE"),
        "shape_area": props.get("Shape_Area"),
        "shape_leng": props.get("Shape_Leng"),
        "geometry": geom,
        "shapely_geometry": shape(geom),
    })

admin_df = pd.DataFrame(admin_records)

print("ADM2 records loaded:", len(admin_df))
print("Geometry types:", admin_df["shapely_geometry"].apply(lambda g: g.geom_type).value_counts().to_dict())
print("\nSmallest ADM2 polygons:")
print(
    admin_df
    .sort_values("shape_area")
    [["adm1_name", "adm2_name", "adm2_code", "shape_area"]]
    .head(10)
)

ADM2 records loaded: 62
Geometry types: {'Polygon': 58, 'GeometryCollection': 4}

Smallest ADM2 polygons:
              adm1_name    adm2_name  adm2_code  shape_area
0                Harare  Chitungwiza      68807    0.004073
39             Bulawayo     Bulawayo      33051    0.039871
1                Harare       Harare      68809    0.081807
7   Mashonaland Central      Bindura      33060    0.192696
11  Mashonaland Central     Rushinga      33065    0.196443
60             Midlands   Zvishavane      33107    0.212085
28     Mashonaland East    Goromonzi      33067    0.212498
5            Manicaland       Mutasa      33058    0.212884
29     Mashonaland East       Hwedza      33068    0.219286
34     Mashonaland East         Seke      33073    0.225164


In [5]:
# Cell 4: Normalize ADM2 geometries to Polygon/MultiPolygon only

from shapely.geometry import GeometryCollection, MultiPolygon, Polygon
from shapely.ops import unary_union

def to_polygonal_geometry(geom):
    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom

    if isinstance(geom, GeometryCollection):
        polygon_parts = [
            part for part in geom.geoms
            if isinstance(part, (Polygon, MultiPolygon))
        ]

        if not polygon_parts:
            return None

        merged = unary_union(polygon_parts)

        if isinstance(merged, (Polygon, MultiPolygon)):
            return merged

    return None


admin_df["polygon_geometry"] = admin_df["shapely_geometry"].apply(to_polygonal_geometry)
admin_df["is_polygon_usable"] = admin_df["polygon_geometry"].notna()
admin_df["polygon_area"] = admin_df["polygon_geometry"].apply(
    lambda g: g.area if g is not None else None
)

usable_admin_df = (
    admin_df[admin_df["is_polygon_usable"]]
    .sort_values("polygon_area")
    .reset_index(drop=True)
    .copy()
)

print("Original ADM2 records:", len(admin_df))
print("Usable polygon ADM2 records:", len(usable_admin_df))
print("Dropped records:", len(admin_df) - len(usable_admin_df))
print("Usable geometry types:", usable_admin_df["polygon_geometry"].apply(lambda g: g.geom_type).value_counts().to_dict())

print("\nSmallest usable ADM2 polygons:")
print(
    usable_admin_df[
        ["adm1_name", "adm2_name", "adm2_code", "polygon_area"]
    ].head(10)
)

Original ADM2 records: 62
Usable polygon ADM2 records: 62
Dropped records: 0
Usable geometry types: {'Polygon': 62}

Smallest usable ADM2 polygons:
             adm1_name    adm2_name  adm2_code  polygon_area
0               Harare  Chitungwiza      68807      0.004073
1             Bulawayo     Bulawayo      33051      0.039871
2               Harare       Harare      68809      0.081807
3  Mashonaland Central      Bindura      33060      0.192696
4  Mashonaland Central     Rushinga      33065      0.196443
5             Midlands   Zvishavane      33107      0.212085
6     Mashonaland East    Goromonzi      33067      0.212498
7           Manicaland       Mutasa      33058      0.212883
8     Mashonaland East       Hwedza      33068      0.219286
9     Mashonaland East         Seke      33073      0.225164


In [6]:
# Cell 5: Load master inputs and initialize PolygonInferenceService

from app.services.polygon_inference_service import PolygonInferenceService

WORKSPACE_DIR = PROJECT_ROOT.parent

MASTER_PATH = (
    WORKSPACE_DIR
    / "data"
    / "master_inputs"
    / "master_inputs_long_198001_205012.parquet"
)

print("Using master dataset:", MASTER_PATH)

if not MASTER_PATH.exists():
    raise FileNotFoundError(f"Master dataset not found: {MASTER_PATH}")

master_df = pd.read_parquet(MASTER_PATH)

# Keep yyyymm consistent for filtering and service use.
master_df["yyyymm"] = master_df["yyyymm"].astype(str)

service = PolygonInferenceService(master_df=master_df)

print("Master rows:", len(master_df))
print("Master columns:", len(master_df.columns))
print("Scenarios:", sorted(master_df["scenario"].dropna().unique().tolist()))
print("Month range:", master_df["yyyymm"].min(), "→", master_df["yyyymm"].max())
print("PolygonInferenceService initialized.")

Using master dataset: C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_long_198001_205012.parquet
Master rows: 38957625
Master columns: 13
Scenarios: ['historical', 'ssp245', 'ssp370', 'ssp585']
Month range: 198001 → 205012
PolygonInferenceService initialized.


In [7]:
# Cell 6: Hardcode drought events and expand to available monthly targets

DROUGHT_EVENTS = [
    {"event_id": 1, "period": "1902-1903", "season": "1902/03", "start_year": 1902, "end_year": 1903, "duration_months": 12, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 2, "period": "1911-1912", "season": "1911/12", "start_year": 1911, "end_year": 1912, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 3, "period": "1921-1922", "season": "1921/22", "start_year": 1921, "end_year": 1922, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 4, "period": "1932-1933", "season": "1932/33", "start_year": 1932, "end_year": 1933, "duration_months": 12, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 5, "period": "1946-1947", "season": "1946/47", "start_year": 1946, "end_year": 1947, "duration_months": 11, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 6, "period": "1967-1968", "season": "1967/68", "start_year": 1967, "end_year": 1968, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 7, "period": "1972-1973", "season": "1972/73", "start_year": 1972, "end_year": 1973, "duration_months": 11, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 8, "period": "1982-1984", "season": "1982-84", "start_year": 1982, "end_year": 1984, "duration_months": 24, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 9, "period": "1986-1987", "season": "1986/87", "start_year": 1986, "end_year": 1987, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 10, "period": "1991-1992", "season": "1991/92", "start_year": 1991, "end_year": 1992, "duration_months": 12, "timeline_severity": 5, "severity_label": "5 - Catastrophic"},
    {"event_id": 11, "period": "1994-1995", "season": "1994/95", "start_year": 1994, "end_year": 1995, "duration_months": 10, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 12, "period": "1997-1998", "season": "1997/98", "start_year": 1997, "end_year": 1998, "duration_months": 12, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 13, "period": "2001-2002", "season": "2001/02 + 2002/03", "start_year": 2001, "end_year": 2002, "duration_months": 18, "timeline_severity": 4, "severity_label": "4 - Extreme"},
    {"event_id": 14, "period": "2012-2013", "season": "2012/13", "start_year": 2012, "end_year": 2013, "duration_months": 8, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 15, "period": "2015-2016", "season": "2015/16", "start_year": 2015, "end_year": 2016, "duration_months": 12, "timeline_severity": 4, "severity_label": "4 - Extreme"},
    {"event_id": 16, "period": "2018-2019", "season": "2018/19", "start_year": 2018, "end_year": 2019, "duration_months": 8, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 17, "period": "2023-2024", "season": "2023/24", "start_year": 2023, "end_year": 2024, "duration_months": 12, "timeline_severity": 5, "severity_label": "5 - Catastrophic"},
]

available_months = set(master_df["yyyymm"].unique())

monthly_rows = []

for event in DROUGHT_EVENTS:
    for year in range(event["start_year"], event["end_year"] + 1):
        for month in range(1, 13):
            yyyymm = f"{year}{month:02d}"
            monthly_rows.append({
                **event,
                "year": year,
                "month": month,
                "yyyymm": yyyymm,
                "available_in_master": yyyymm in available_months,
            })

backtest_months_df = pd.DataFrame(monthly_rows)

available_backtest_months_df = (
    backtest_months_df[backtest_months_df["available_in_master"]]
    .reset_index(drop=True)
    .copy()
)

print("Total expanded event-month rows:", len(backtest_months_df))
print("Available event-month rows:", len(available_backtest_months_df))
print("Available periods:", available_backtest_months_df["period"].drop_duplicates().tolist())

print("\nFirst available rows:")
print(available_backtest_months_df.head(12))

Total expanded event-month rows: 420
Available event-month rows: 252
Available periods: ['1982-1984', '1986-1987', '1991-1992', '1994-1995', '1997-1998', '2001-2002', '2012-2013', '2015-2016', '2018-2019', '2023-2024']

First available rows:
    event_id     period   season  start_year  end_year  duration_months  \
0          8  1982-1984  1982-84        1982      1984               24   
1          8  1982-1984  1982-84        1982      1984               24   
2          8  1982-1984  1982-84        1982      1984               24   
3          8  1982-1984  1982-84        1982      1984               24   
4          8  1982-1984  1982-84        1982      1984               24   
5          8  1982-1984  1982-84        1982      1984               24   
6          8  1982-1984  1982-84        1982      1984               24   
7          8  1982-1984  1982-84        1982      1984               24   
8          8  1982-1984  1982-84        1982      1984               24   
9       

In [8]:
# Cell 7: Define drought class and severity comparison helpers

CLASS_TO_SEVERITY = {
    "normal": 0,
    "moderate": 2,
    "severe": 3,
    "extreme": 4,
}

SEVERITY_TO_LABEL = {
    0: "0 - Normal",
    1: "1 - Mild",
    2: "2 - Moderate",
    3: "3 - Severe",
    4: "4 - Extreme",
    5: "5 - Catastrophic",
}

def model_class_to_severity(model_class: str | None) -> int | None:
    if model_class is None:
        return None

    return CLASS_TO_SEVERITY.get(str(model_class).lower())


def compare_timeline_and_model(timeline_severity: int, model_class: str | None) -> dict:
    model_severity = model_class_to_severity(model_class)

    if model_severity is None:
        return {
            "model_severity": None,
            "model_severity_label": None,
            "severity_gap": None,
            "comparison": "missing_model_class",
        }

    severity_gap = model_severity - int(timeline_severity)

    if severity_gap == 0:
        comparison = "matched"
    elif severity_gap < 0:
        comparison = "underestimated"
    else:
        comparison = "overestimated"

    return {
        "model_severity": model_severity,
        "model_severity_label": SEVERITY_TO_LABEL.get(model_severity),
        "severity_gap": severity_gap,
        "comparison": comparison,
    }


print("Class-to-severity mapping:")
print(CLASS_TO_SEVERITY)

print("\nExample comparison:")
print(compare_timeline_and_model(5, "moderate"))

Class-to-severity mapping:
{'normal': 0, 'moderate': 2, 'severe': 3, 'extreme': 4}

Example comparison:
{'model_severity': 2, 'model_severity_label': '2 - Moderate', 'severity_gap': -3, 'comparison': 'underestimated'}


In [9]:
# Cell 8: Safe rural ADM2 smoke test for one drought month

import gc
import time

SCENARIO = "historical"
MODEL = "hybrid"

# Start with a rural drought-sensitive province/district candidate.
TEST_ADM2_NAME = "Chivi"
TEST_YYYYMM = "202310"

admin_match = usable_admin_df[usable_admin_df["adm2_name"].eq(TEST_ADM2_NAME)]

if admin_match.empty:
    raise ValueError(
        f"Could not find ADM2 '{TEST_ADM2_NAME}'. "
        f"Available examples: {usable_admin_df['adm2_name'].head(20).tolist()}"
    )

event_match = available_backtest_months_df[
    available_backtest_months_df["yyyymm"].eq(TEST_YYYYMM)
]

if event_match.empty:
    raise ValueError(f"No drought event found for {TEST_YYYYMM}")

admin_row = admin_match.iloc[0]
event_row = event_match.iloc[0]

print("Smoke test scope")
print("Admin:", admin_row[["adm1_name", "adm2_name", "adm2_code", "polygon_area"]].to_dict())
print("Month:", TEST_YYYYMM)
print("Timeline severity:", event_row["timeline_severity"], event_row["severity_label"])

started = time.time()

try:
    print("\nRunning polygon inference...")
    result = service.infer_polygon(
        geometry=admin_row["polygon_geometry"],
        scenario=SCENARIO,
        yyyymm=int(TEST_YYYYMM),
        model=MODEL,
    )

    comparison = compare_timeline_and_model(
        timeline_severity=int(event_row["timeline_severity"]),
        model_class=result.summary.get("dominant_class"),
    )

    smoke_test_result = {
        "adm1_name": admin_row["adm1_name"],
        "adm2_name": admin_row["adm2_name"],
        "adm2_code": admin_row["adm2_code"],
        "yyyymm": TEST_YYYYMM,
        "timeline_severity": int(event_row["timeline_severity"]),
        "severity_label": event_row["severity_label"],
        "cells_inferred": result.summary.get("cells_inferred"),
        "mean_confidence": result.summary.get("mean_confidence"),
        "dominant_class": result.summary.get("dominant_class"),
        **comparison,
        "elapsed_seconds": round(time.time() - started, 2),
    }

    print("Success.")
    print(smoke_test_result)

except Exception as exc:
    smoke_test_result = {
        "adm2_name": TEST_ADM2_NAME,
        "yyyymm": TEST_YYYYMM,
        "status": "failed",
        "error_type": type(exc).__name__,
        "error": str(exc),
        "elapsed_seconds": round(time.time() - started, 2),
    }

    print("Failed.")
    print(smoke_test_result)

finally:
    gc.collect()

Smoke test scope
Admin: {'adm1_name': 'Masvingo', 'adm2_name': 'Chivi', 'adm2_code': 33083, 'polygon_area': 0.3140117051271317}
Month: 202310
Timeline severity: 5 5 - Catastrophic

Running polygon inference...

Success.
{'adm1_name': 'Masvingo', 'adm2_name': 'Chivi', 'adm2_code': np.int64(33083), 'yyyymm': '202310', 'timeline_severity': 5, 'severity_label': '5 - Catastrophic', 'cells_inferred': 154, 'mean_confidence': 0.44631829683656815, 'dominant_class': 'severe', 'model_severity': 3, 'model_severity_label': '3 - Severe', 'severity_gap': -2, 'comparison': 'underestimated', 'elapsed_seconds': 237.37}


In [10]:
# Cell 9: Controlled ADM2 backtest batch for selected rural districts

import gc
import time
from pathlib import Path

SCENARIO = "historical"
MODEL = "hybrid"

SELECTED_ADM2_NAMES = [
    "Chivi",
    "Mwenezi",
    "Beitbridge",
    "Gwanda",
    "Buhera",
]

SELECTED_MONTHS = [
    "202310",
    "202311",
    "202312",
    "202401",
    "202402",
    "202403",
    "202404",
]

MAX_ADMINS = 5
MAX_MONTHS = 7

batch_admin_df = (
    usable_admin_df[usable_admin_df["adm2_name"].isin(SELECTED_ADM2_NAMES)]
    .head(MAX_ADMINS)
    .reset_index(drop=True)
    .copy()
)

batch_month_df = (
    available_backtest_months_df[
        available_backtest_months_df["yyyymm"].isin(SELECTED_MONTHS)
    ]
    .sort_values("yyyymm")
    .head(MAX_MONTHS)
    .reset_index(drop=True)
    .copy()
)

print("Controlled batch scope")
print("Admins:", len(batch_admin_df), batch_admin_df["adm2_name"].tolist())
print("Months:", len(batch_month_df), batch_month_df["yyyymm"].tolist())
print("Estimated service calls:", len(batch_admin_df) * len(batch_month_df))

batch_rows = []
total_calls = len(batch_admin_df) * len(batch_month_df)
call_i = 0

for _, admin_row in batch_admin_df.iterrows():
    for _, event_row in batch_month_df.iterrows():
        call_i += 1
        started = time.time()

        adm2_name = admin_row["adm2_name"]
        yyyymm = event_row["yyyymm"]

        print(
            f"\n[{call_i}/{total_calls}] Running "
            f"{admin_row['adm1_name']} / {adm2_name} | {yyyymm}"
        )

        try:
            result = service.infer_polygon(
                geometry=admin_row["polygon_geometry"],
                scenario=SCENARIO,
                yyyymm=int(yyyymm),
                model=MODEL,
            )

            comparison = compare_timeline_and_model(
                timeline_severity=int(event_row["timeline_severity"]),
                model_class=result.summary.get("dominant_class"),
            )

            row = {
                "adm1_name": admin_row["adm1_name"],
                "adm2_name": adm2_name,
                "adm2_code": int(admin_row["adm2_code"]),
                "yyyymm": yyyymm,
                "event_id": int(event_row["event_id"]),
                "period": event_row["period"],
                "season": event_row["season"],
                "timeline_severity": int(event_row["timeline_severity"]),
                "severity_label": event_row["severity_label"],
                "status": "success",
                "cells_inferred": result.summary.get("cells_inferred"),
                "mean_confidence": result.summary.get("mean_confidence"),
                "dominant_class": result.summary.get("dominant_class"),
                **comparison,
                "elapsed_seconds": round(time.time() - started, 2),
                "error": None,
            }

            print(
                "Success | "
                f"cells={row['cells_inferred']} | "
                f"class={row['dominant_class']} | "
                f"gap={row['severity_gap']} | "
                f"seconds={row['elapsed_seconds']}"
            )

        except Exception as exc:
            row = {
                "adm1_name": admin_row["adm1_name"],
                "adm2_name": adm2_name,
                "adm2_code": int(admin_row["adm2_code"]),
                "yyyymm": yyyymm,
                "event_id": int(event_row["event_id"]),
                "period": event_row["period"],
                "season": event_row["season"],
                "timeline_severity": int(event_row["timeline_severity"]),
                "severity_label": event_row["severity_label"],
                "status": "failed",
                "cells_inferred": None,
                "mean_confidence": None,
                "dominant_class": None,
                "model_severity": None,
                "model_severity_label": None,
                "severity_gap": None,
                "comparison": "failed",
                "elapsed_seconds": round(time.time() - started, 2),
                "error": f"{type(exc).__name__}: {exc}",
            }

            print(
                "Failed | "
                f"{row['error']} | "
                f"seconds={row['elapsed_seconds']}"
            )

        batch_rows.append(row)
        gc.collect()

controlled_batch_df = pd.DataFrame(batch_rows)

print("\nControlled batch complete")
print(controlled_batch_df)

print("\nComparison summary")
print(
    controlled_batch_df
    .groupby(["adm1_name", "adm2_name", "comparison"], dropna=False)
    .size()
    .reset_index(name="months")
)

Controlled batch scope
Admins: 5 ['Chivi', 'Buhera', 'Gwanda', 'Beitbridge', 'Mwenezi']
Months: 7 ['202310', '202311', '202312', '202401', '202402', '202403', '202404']
Estimated service calls: 35

[1/35] Running Masvingo / Chivi | 202310
Success | cells=154 | class=severe | gap=-2 | seconds=271.51

[2/35] Running Masvingo / Chivi | 202311
Success | cells=154 | class=normal | gap=-5 | seconds=355.24

[3/35] Running Masvingo / Chivi | 202312
Success | cells=154 | class=normal | gap=-5 | seconds=352.49

[4/35] Running Masvingo / Chivi | 202401
Success | cells=154 | class=normal | gap=-5 | seconds=368.19

[5/35] Running Masvingo / Chivi | 202402
Success | cells=154 | class=normal | gap=-5 | seconds=296.35

[6/35] Running Masvingo / Chivi | 202403
Success | cells=154 | class=normal | gap=-5 | seconds=286.39

[7/35] Running Masvingo / Chivi | 202404
Success | cells=154 | class=normal | gap=-5 | seconds=282.93

[8/35] Running Manicaland / Buhera | 202310
Success | cells=230 | class=severe | 

In [11]:
# Cell 10: Save controlled backtest outputs

from pathlib import Path

BACKTEST_OUTPUT_DIR = PROJECT_ROOT.parent / "data" / "backtests"
BACKTEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROLLED_BATCH_CSV = BACKTEST_OUTPUT_DIR / "controlled_adm2_backtest_2023_2024.csv"
CONTROLLED_BATCH_PARQUET = BACKTEST_OUTPUT_DIR / "controlled_adm2_backtest_2023_2024.parquet"

controlled_batch_df.to_csv(CONTROLLED_BATCH_CSV, index=False)
controlled_batch_df.to_parquet(CONTROLLED_BATCH_PARQUET, index=False)

print("Saved controlled backtest outputs:")
print("CSV:", CONTROLLED_BATCH_CSV)
print("Parquet:", CONTROLLED_BATCH_PARQUET)
print("Rows saved:", len(controlled_batch_df))

Saved controlled backtest outputs:
CSV: C:\Projects\Infer RozviDrought\data\backtests\controlled_adm2_backtest_2023_2024.csv
Parquet: C:\Projects\Infer RozviDrought\data\backtests\controlled_adm2_backtest_2023_2024.parquet
Rows saved: 35
